# Data Preprocessing for taiko-siffusion

## Unpack .osz Files

Unpack all `.osz` files from `sample_data/raw/*.osz` into `sample_data/unpacked/`.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
from src.preprocessing.unpack_osz import unpack_osz_files

source_glob = "sample_data/raw/*.osz"
destination_root = "sample_data/unpacked"

extracted = unpack_osz_files(
    source_glob=source_glob,
    destination_root=destination_root,
    overwrite=False,
)

print(f"Processed {len(extracted)} archive(s).")

Unpacking .osz files: 100%|██████████| 5/5 [00:00<00:00, 12.78file/s]

Processed 5 archive(s).


## Process into intermediate files

Below is a sample that parses one selected beatmap, then reconstruct the beatmap using the same parsed files. The parsed *.notes.json files should be used for training along with the corresponding *.mp3 or *.ogg file.

The map picked in this example is: 2516534 OU - Mountain (FORMless000) [dang dang di dang]

In [ ]:
from pathlib import Path
from src.preprocessing.osutaiko_parser import parse_osu_file_to_jsons
map_name = "OU - Mountain (FORMless000) [dang dang di dang]"
map_id = "2516534"
osu_path = Path(f"sample_data/unpacked/{map_id}/{map_name}.osu")
out_dir = Path(f"sample_data/unpacked/{map_id}/parsed")

parse_osu_file_to_jsons(
    osu_path=osu_path,
    out_dir=out_dir,
    include_bpm_events=True,
)

In [4]:
# Inspections

import json

notes_path = out_dir / f"{map_name}.notes.json"
timing_path = out_dir / f"{map_name}.timing.json"
metadata_path = out_dir / f"{map_name}.metadata.json"

with notes_path.open("r", encoding="utf-8") as f:
    notes = json.load(f)

with timing_path.open("r", encoding="utf-8") as f:
    timing = json.load(f)

with metadata_path.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

print("First 10 note events:")
for event in notes["notes"][:10]:
    print(event)

print("\nFirst 5 timing points:")
for tp in timing["timing_points"][:5]:
    print(tp)

print("\nMetadata title:", metadata["metadata"]["Title"])

# Check BPM change events

bpm_events = [n for n in notes["notes"] if n["type"] == "bpmchange"]
print(f"Found {len(bpm_events)} bpmchange events")
bpm_events[:5]

First 10 note events:
{'type': 'bpmchange', 'time': 483, 'raw_time': 483, 'sv': 196.87499999999983, 'volume': 77, 'bpm': 140.6249999999999, 'meter': 4}
{'type': 'don', 'time': 483, 'raw_time': 483, 'sv': 196.87499999999983, 'volume': 77, 'bpm': None, 'meter': None}
{'type': 'kat', 'time': 696.3333333333335, 'raw_time': 696, 'sv': 196.87499999999983, 'volume': 77, 'bpm': None, 'meter': None}
{'type': 'kat', 'time': 909.666666666667, 'raw_time': 909, 'sv': 196.87499999999983, 'volume': 77, 'bpm': None, 'meter': None}
{'type': 'don', 'time': 1016.3333333333338, 'raw_time': 1016, 'sv': 196.87499999999983, 'volume': 77, 'bpm': None, 'meter': None}
{'type': 'kat', 'time': 1123.0000000000005, 'raw_time': 1123, 'sv': 196.87499999999983, 'volume': 77, 'bpm': None, 'meter': None}
{'type': 'kat', 'time': 1549.6666666666677, 'raw_time': 1549, 'sv': 196.87499999999983, 'volume': 77, 'bpm': None, 'meter': None}
{'type': 'don', 'time': 1763.0000000000011, 'raw_time': 1763, 'sv': 196.87499999999983, '

[{'type': 'bpmchange',
  'time': 483,
  'raw_time': 483,
  'sv': 196.87499999999983,
  'volume': 77,
  'bpm': 140.6249999999999,
  'meter': 4}]

## Reconstruct beatmaps from parsed files

In [6]:
from pathlib import Path
from src.preprocessing.osutaiko_reconstructor import reconstruct_osu

# Full metadata test

notes_path = Path(out_dir / f"{map_name}.notes.json")
timing_path = Path(out_dir / f"{map_name}.timing.json")
metadata_path = Path(out_dir / f"{map_name}.metadata.json")
out_path = Path(out_dir / f"{map_name}.reconstructed.osu")

reconstruct_osu(
    notes_path=notes_path,
    timing_path=timing_path,
    metadata_path=metadata_path,
    out_path=out_path,
)

print(out_path.read_text(encoding="utf-8")[:2000])

# Note data only test
notes_only_out = Path(out_dir / f"{map_name}.notes_only.reconstructed.osu")

reconstruct_osu(
    notes_path=notes_path,
    timing_path=None,
    metadata_path=None,
    out_path=notes_only_out,
)

print(notes_only_out.read_text(encoding="utf-8")[:2000])

# Check BPM change events

import json

with notes_path.open("r", encoding="utf-8") as f:
    notes_json = json.load(f)

bpm_events = [n for n in notes_json["notes"] if n["type"] == "bpmchange"]
print("BPM events in notes:", len(bpm_events))
print("Example:", bpm_events[:3])

osu file format v14

[General]
AudioFilename:audio.mp3
AudioLeadIn:0
PreviewTime:468
Countdown:0
SampleSet:Normal
StackLeniency:0.7
Mode:1
LetterboxInBreaks:0
WidescreenStoryboard:1

[Editor]
DistanceSpacing:1
BeatDivisor:4
GridSize:32
TimelineZoom:1

[Metadata]
Title:MORNING OF PUYOPUYO
TitleUnicode:MORNING OF PUYOPUYO
Artist:Masanobu Tsukamoto
ArtistUnicode:塚本雅信
Creator:Peaceful
Version:Oni
Source:ぷよぷよ
Tags:MetalStream Video Game Instrumental Sega Genesis Mega Drive Compile original soundtrack vgm practice mode Tokoton from music puyo mats ost
BeatmapID:5014652
BeatmapSetID:2335985

[Difficulty]
HPDrainRate:7.5
CircleSize:5
OverallDifficulty:5
ApproachRate:9
SliderMultiplier:1.4
SliderTickRate:1.0

[Events]
//Background and Video events
//Break Periods

[TimingPoints]
483,426.666666666667,4,1,0,77,1,0

[HitObjects]
256,192,483,1,0,0:0:0:0:
256,192,696,1,8,0:0:0:0:
256,192,910,1,8,0:0:0:0:
256,192,1016,1,0,0:0:0:0:
256,192,1123,1,8,0:0:0:0:
256,192,1550,1,8,0:0:0:0:
256,192,1763,1,0,0